---
## Model B - CNN + ERA5 Classifier (ResNet-18 Backbone)
Switched to ImageNet-pretrained ResNet-18 (replaces from-scratch CNN that collapsed to majority class).
Focal loss + class weights + augmentation. lr=1e-4, patience=10, 50 epochs max.

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('WARNING: No GPU found. Enable GPU in Notebook Settings.')

In [ ]:
# ── Cell 2: Install extra dependencies ───────────────────────────────────
!pip install netCDF4==1.6.5 opencv-python-headless==4.9.0.80 -q

In [ ]:
import os, shutil, sys
from pathlib import Path

WORK = Path('/kaggle/working')

# Auto-detect actual dataset mount paths (handles both short and long Kaggle paths)
def find_dataset(name):
    """Search /kaggle/input recursively for a dataset folder by name."""
    base = Path('/kaggle/input')
    # Try direct path first (short form)
    if (base / name).exists():
        return base / name
    # Search up to 4 levels deep for the folder
    for p in sorted(base.rglob(name)):
        if p.is_dir():
            return p
    # Fallback: search for any folder containing the name
    for p in sorted(base.rglob('*')):
        if p.is_dir() and name in p.name:
            return p
    return None

DATA_INPUT = find_dataset('cyclovision-data')
SRC_INPUT  = find_dataset('cyclovision-src')

print('cyclovision-data found at:', DATA_INPUT)
print('cyclovision-src  found at:', SRC_INPUT)

if DATA_INPUT is None:
    raise RuntimeError('cyclovision-data dataset not found! Add it in the right sidebar.')
if SRC_INPUT is None:
    raise RuntimeError('cyclovision-src dataset not found! Add it in the right sidebar.')

# Copy src/
if (WORK / 'src').exists():
    shutil.rmtree(WORK / 'src')
src_dir = SRC_INPUT / 'src' if (SRC_INPUT / 'src').exists() else SRC_INPUT
shutil.copytree(str(src_dir), str(WORK / 'src'))
print('Copied src/ OK -', len(list((WORK/'src').rglob('*.py'))), 'Python files')

# Figure out where raw data sits (handle raw/ subdirectory or flat)
DATA_RAW = DATA_INPUT / 'raw' if (DATA_INPUT / 'raw').exists() else DATA_INPUT

def copy_dir(src, dst):
    dst = Path(dst)
    if dst.exists(): shutil.rmtree(dst)
    shutil.copytree(str(src), str(dst))
    n = sum(1 for _ in dst.rglob('*') if _.is_file())
    print(f'  {dst.name}: {n} files')

for subdir in ['data/raw/insat', 'data/raw/era5', 'data/raw/ibtracs']:
    (WORK / subdir).mkdir(parents=True, exist_ok=True)

copy_dir(DATA_RAW / 'insat',   WORK / 'data/raw/insat')
copy_dir(DATA_RAW / 'era5',    WORK / 'data/raw/era5')
copy_dir(DATA_RAW / 'ibtracs', WORK / 'data/raw/ibtracs')

sys.path.insert(0, str(WORK))
os.chdir(str(WORK))
print('Working directory:', os.getcwd())


In [ ]:
# ── Cell 4: Verify data ───────────────────────────────────────────────────
from src.config import INSAT_DIR, ERA5_DIR, IBTRACS_DIR
from src.data.load_besttrack import load_observations

obs = load_observations()
print(f'Best-track observations : {len(obs):,} rows')
print(f'Unique storms           : {obs["storm_id"].nunique()}')

insat_storms = {p.name: len(list(p.glob('*.npy')))
                for p in INSAT_DIR.iterdir() if p.is_dir()}
total_frames = sum(insat_storms.values())
print(f'\nINSAT .npy frames : {total_frames:,} across {len(insat_storms)} storms')
for sid, n in sorted(insat_storms.items()):
    print(f'  {sid}: {n}')

assert total_frames >= 13000, f'Expected ~13479 frames, got {total_frames} -- check upload'
print(f'\nERA5 files: {len(list(ERA5_DIR.glob("*.nc")))}')
print('Data verification passed.')

---
## Model A — U-Net Detector
Detects the cyclone vortex in INSAT satellite imagery.

In [ ]:
# ── Cell 5: Delete old detector checkpoint ───────────────────────────────
det_ckpt = WORK / 'detector_checkpoint.pt'
if det_ckpt.exists():
    det_ckpt.unlink()
    print('Deleted old detector_checkpoint.pt')
else:
    print('No existing detector checkpoint -- training fresh')

In [ ]:
# ── Cell 6: Train Model A (Detector) ─────────────────────────────────────
# Expected: ~15-20 min on T4 for 30 epochs over 13,479 frames
!cd /kaggle/working && python -m src.training.train_detector \
    --epochs 30 \
    --batch-size 32 \
    --lr 1e-3 \
    --val-split 0.2 \
    --base-channels 16

In [ ]:
# ── Cell 7: Verify detector checkpoint ───────────────────────────────────
p = WORK / 'detector_checkpoint.pt'
if p.exists():
    print(f'detector_checkpoint.pt  {p.stat().st_size/1e6:.1f} MB  OK')
else:
    raise RuntimeError('detector_checkpoint.pt not found -- check Cell 6 output')

---
## Model B - CNN + ERA5 Classifier (ResNet-18 Backbone)
Switched to ImageNet-pretrained ResNet-18 (replaces from-scratch CNN that collapsed to majority class).
Focal loss + class weights + augmentation. lr=1e-4, patience=10, 50 epochs max.

In [ ]:
# ── Cell 8: Delete old classifier checkpoint ─────────────────────────────
cls_ckpt = WORK / 'classifier_checkpoint.pt'
if cls_ckpt.exists():
    cls_ckpt.unlink()
    print('Deleted old classifier_checkpoint.pt')
else:
    print('No existing classifier checkpoint -- training fresh')

In [ ]:
# Model B: ResNet-18 backbone inlined -- avoids needing updated src upload
import sys, os, time, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from pathlib import Path

sys.path.insert(0, '/kaggle/working')
os.chdir('/kaggle/working')

from src.config import NUM_CATEGORIES, ERA5_FEATURES, CATEGORY_NAMES
from src.data.dataset import CycloneClassificationDataset
from src.models.classification import CycloneClassifier, build_pretrained_backbone
from src.training.metrics import per_class_f1, confusion_matrix as conf_matrix
from collections import defaultdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- Hyperparams ---
LR, WD, PATIENCE, EPOCHS, BATCH = 1e-4, 1e-4, 10, 50, 32
VAL_SPLIT = 0.2

# --- Dataset ---
full_ds = CycloneClassificationDataset()
base_ds = full_ds
while hasattr(base_ds, 'dataset'): base_ds = base_ds.dataset

records = base_ds._real_records
storm_indices = defaultdict(list)
for idx, r in enumerate(records):
    storm_indices[r['storm_id']].append(idx)

train_idx, val_idx = [], []
for sid, idxs in storm_indices.items():
    cut = max(1, int(len(idxs) * (1 - VAL_SPLIT)))
    train_idx.extend(idxs[:cut])
    val_idx.extend(idxs[cut:])

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} -- per-storm 80/20 split')

# --- Class weights ---
labels = np.array([r['category'] for r in records])
counts = np.bincount(labels, minlength=NUM_CATEGORIES).astype(float)
active = [c for c in range(NUM_CATEGORIES) if counts[c] > 0]
weights = np.zeros(NUM_CATEGORIES, dtype=np.float32)
ac = counts[active]
aw = ac.sum() / (len(active) * ac)
for i, c in enumerate(active): weights[c] = aw[i]
class_weights = torch.tensor(weights).to(device)
print('Active classes:', [CATEGORY_NAMES[c] for c in active])

# --- Focal Loss ---
class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=2.0, ls=0.1):
        super().__init__()
        self.w, self.g, self.ls = weight, gamma, ls
    def forward(self, logits, targets):
        ce = nn.functional.cross_entropy(logits, targets, weight=self.w,
                                          label_smoothing=self.ls, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.g) * ce).mean()

# --- Model: ResNet-18 pretrained backbone ---
backbone = build_pretrained_backbone('resnet18', embed_dim=128, in_channels=1)
model = CycloneClassifier(num_categories=NUM_CATEGORIES,
                          num_era5_features=len(ERA5_FEATURES),
                          image_backbone=backbone).to(device)
print('Model: ResNet-18 backbone')

criterion = FocalLoss(class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

train_loader = DataLoader(Subset(full_ds, train_idx), batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(Subset(full_ds, val_idx),   batch_size=BATCH, shuffle=False, num_workers=2)

best_loss, no_imp = float('inf'), 0
y_true_best = y_pred_best = np.array([])

print(f'Training: lr={LR}, wd={WD}, patience={PATIENCE}, epochs={EPOCHS}')
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    model.train()
    for batch in train_loader:
        imgs = batch['image'].to(device)
        era5 = batch['era5'].to(device)
        lbls = batch['label'].to(device)
        loss = criterion(model(imgs, era5), lbls)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    vl, yt, yp = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image'].to(device)
            era5 = batch['era5'].to(device)
            lbls = batch['label'].to(device)
            logits = model(imgs, era5)
            vl += criterion(logits, lbls).item() * imgs.size(0)
            yt.extend(lbls.cpu().numpy()); yp.extend(logits.argmax(1).cpu().numpy())
    val_loss = vl / len(val_loader.dataset)
    yt, yp = np.array(yt), np.array(yp)
    acc = (yt == yp).mean()
    scheduler.step(val_loss)

    mark = ''
    if val_loss < best_loss:
        best_loss = val_loss; no_imp = 0
        torch.save(model.state_dict(), 'classifier_checkpoint.pt')
        mark = ' * [Saved]'
        y_true_best, y_pred_best = yt.copy(), yp.copy()
    else:
        no_imp += 1

    print(f'Epoch {epoch:02d}/{EPOCHS} ({time.time()-t0:.1f}s) | val_loss={val_loss:.4f} val_acc={acc:.1%}{mark}')
    if no_imp >= PATIENCE:
        print(f'Early stopping at epoch {epoch}'); break

print('\nFinal per-class F1:')
f1s = per_class_f1(y_true_best, y_pred_best, num_classes=NUM_CATEGORIES)
for idx, f1 in f1s.items():
    sup = int((y_true_best == idx).sum())
    if sup > 0:
        print(f'  {CATEGORY_NAMES[idx]:<35} F1={f1:.2f}  (support={sup})')
print('\nClassifier training done. Saved to classifier_checkpoint.pt')


In [ ]:
# ── Cell 10: Verify classifier + print F1 ────────────────────────────────
p = WORK / 'classifier_checkpoint.pt'
if p.exists():
    print(f'classifier_checkpoint.pt  {p.stat().st_size/1e6:.1f} MB  OK')
else:
    raise RuntimeError('classifier_checkpoint.pt not found -- check Cell 9 output')

# Quick F1 check on val set
import torch, json
from src.models.classification import CycloneClassifier
from src.config import NUM_CATEGORIES, ERA5_FEATURES
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(p, map_location=device)
if isinstance(ckpt, dict) and 'val_f1' in ckpt:
    print(f'Best val F1 : {ckpt["val_f1"]:.4f}')
elif isinstance(ckpt, dict) and 'metrics' in ckpt:
    print('Metrics:', json.dumps(ckpt['metrics'], indent=2))
else:
    print('Checkpoint saved (metrics not embedded -- check training log above)')

---
## Model C — ConvLSTM Predictor
Forecasts cyclone track and intensity from a sequence of past satellite frames.

In [ ]:
# ── Cell 11: Delete old predictor checkpoint ──────────────────────────────
pred_ckpt = WORK / 'predictor_checkpoint.pt'
if pred_ckpt.exists():
    pred_ckpt.unlink()
    print('Deleted old predictor_checkpoint.pt')
else:
    print('No existing predictor checkpoint -- training fresh')

In [ ]:
# ── Cell 12: Train Model C (Predictor) ───────────────────────────────────
# Expected: ~25-40 min on T4 for 30 epochs
# hidden-channels=32 gives better capacity; lr=5e-4 is already the fixed default
!cd /kaggle/working && python -m src.training.train_predictor \
    --epochs 30 \
    --batch-size 16 \
    --lr 5e-4 \
    --hidden-channels 32 \
    --patience 5 \
    --val-split 0.2

In [ ]:
# ── Cell 13: Verify predictor checkpoint ─────────────────────────────────
p = WORK / 'predictor_checkpoint.pt'
if p.exists():
    print(f'predictor_checkpoint.pt  {p.stat().st_size/1e6:.1f} MB  OK')
else:
    raise RuntimeError('predictor_checkpoint.pt not found -- check Cell 12 output')

---
## Final Summary and Download

In [ ]:
# ── Cell 14: Summary and download links ──────────────────────────────────
from IPython.display import display, FileLink

checkpoints = [
    'detector_checkpoint.pt',
    'classifier_checkpoint.pt',
    'predictor_checkpoint.pt',
]

print('='*60)
print('TRAINING COMPLETE')
print('='*60)
all_ok = True
for ckpt in checkpoints:
    path = WORK / ckpt
    if path.exists():
        print(f'  OK  {ckpt:<40} {path.stat().st_size/1e6:.2f} MB')
    else:
        print(f'  MISSING  {ckpt}')
        all_ok = False
print('='*60)

if all_ok:
    print('Download links (click each):')
    for ckpt in checkpoints:
        display(FileLink(ckpt))
    print('Place all three .pt files in:')
    print('  D:\\Hackathon Cyclovision AI\\cyclovision-ai\\')
else:
    print('One or more checkpoints missing -- scroll up to find the error.')

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working')
os.chdir('/kaggle/working')

# ── Cell 15: Sanity check -- load all three models ────────────────────────
import torch
from src.models.detection import CycloneUNet
from src.models.classification import CycloneClassifier
from src.models.prediction import CyclonePredictor
from src.config import NUM_CATEGORIES, ERA5_FEATURES, SEQUENCE_LENGTH_OUT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

det = CycloneUNet(base_channels=16).to(device)  # matches training default
det.load_state_dict(torch.load('detector_checkpoint.pt', map_location=device))
det.eval()
with torch.no_grad():
    mask = det(torch.randn(1, 1, 128, 128).to(device))
print(f'Model A Detector  : output {tuple(mask.shape)}  OK')

cls = CycloneClassifier(num_classes=NUM_CATEGORIES, era5_features=len(ERA5_FEATURES)).to(device)
cls.load_state_dict(torch.load('classifier_checkpoint.pt', map_location=device))
cls.eval()
with torch.no_grad():
    logits = cls(torch.randn(1,1,128,128).to(device), torch.randn(1,len(ERA5_FEATURES)).to(device))
print(f'Model B Classifier: output {tuple(logits.shape)}  OK')

pred = CyclonePredictor(in_channels=1, hidden_channels=32, seq_len_out=SEQUENCE_LENGTH_OUT).to(device)
pred.load_state_dict(torch.load('predictor_checkpoint.pt', map_location=device))
pred.eval()
with torch.no_grad():
    frames, track = pred(torch.randn(1, 8, 1, 64, 64).to(device))
print(f'Model C Predictor : frames {tuple(frames.shape)}, track {tuple(track.shape)}  OK')

print('All three models verified. Ready to download.')